# Dota 2 data acquisition and initial audit

Md Forhadul Islam, London

An independent study adapting the KPI analysis of Dr Fazilat Hojaji, Robert E. McIlroy, Antoine Dupuy, Gianpaolo Pedroni, Adam J. Toth and Mark J. Campbell. Original paper: https://doi.org/10.1016/j.chbr.2025.100718.

Data: Betty Decision Context Dataset, published by wolframko using Valve replays and STRATZ data. https://huggingface.co/datasets/wolframko/betty-dota2. Publisher declared licence: MIT. This notebook does not train models or evaluate a final test set.

## What we want to learn

* Can the original workflow be adapted to identify predictive KPIs in Dota 2?
* Does movement sampling resolution change measurements and conclusions?
* Are KPI rankings stable across models, seeds and samples?
* If the audit supports it, does hero context change their interpretation?

The complete rationale and safeguards are in docs/STUDY_BRIEF.md.


In [1]:
# A module is a collection of reusable Python tools. These imports make those tools available.
from pathlib import Path  # Path builds file locations safely on Windows and other operating systems.
import json  # JSON stores structured results as text so numbers can be checked outside this notebook.
import sys  # sys lets us tell Python where this project's shared code is stored.
import pandas as pd  # pandas organises information into tables; pd is its conventional short name.
from IPython.display import display  # display renders readable tables inside Jupyter.

ROOT = Path.cwd().resolve()  # Start from the folder where the notebook is being run.
if not (ROOT / 'src').is_dir():  # The shared code folder identifies the project root for this notebook.
    ROOT = ROOT.parent  # If we started inside notebooks, move up one folder to the project.
assert (ROOT / 'src').is_dir(), 'Run from the project or notebooks folder.'  # Stop early if paths are wrong.
sys.path.insert(0, str(ROOT / 'src'))  # Search our shared code folder before looking elsewhere.
from audit_dota2 import main as audit  # Reuse the same audit function as the command line script.


## Download provenance

The download script pins the source revision, verifies file sizes and checks publisher SHA256 hashes where supplied. Raw files remain excluded from Git. To download or resume, run `python src/download_dota2.py` from the project folder. This notebook does not automatically fetch the full dataset.


Why verify a download? An incomplete or changed file could alter the study without an obvious error. A checksum is a short fingerprint calculated from the complete file contents. Matching the publisher's checksum checks that we received the same bytes. Pinning a revision means selecting one specific version of the dataset rather than silently following future updates.

A licence describes permitted reuse; it does not guarantee that the dataset is accurate or complete. No models are trained in this notebook.


In [2]:
manifest_path = ROOT / 'outputs/results/D_download_manifest.json'  # Locate the saved download inventory.
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))  # Convert its JSON text into a Python dictionary.
fields = ['source', 'revision', 'declared_licence', 'download_date', 'status', 'expected_files', 'expected_bytes', 'verified_bytes']  # Choose useful provenance fields for the reader.
summary = {key: manifest.get(key) for key in fields}  # Keep only those fields; get returns None for an absent field.
display(pd.Series(summary))  # Show the download's provenance and completion information as a labelled list.
assert manifest['status'] == 'complete', 'Finish or resume the download first.'  # Do not treat a partial dataset as complete.
assert len(manifest['files']) == manifest['expected_files']  # Require one verified entry for every expected source file.
assert not manifest['failures']  # An empty failure list means no source file remains unresolved.


source              https://huggingface.co/datasets/wolframko/bett...
revision                     b20e01577af2f4f5dffcdc4f1d5bad5c3807abca
declared_licence                                                  MIT
download_date                                              2026-10-29
status                                                       complete
expected_files                                                   6824
expected_bytes                                             9654729677
verified_bytes                                             9654729677
dtype: object

## Metadata audit

Count rows using Parquet metadata without loading every table into memory. Detailed completeness checks below cover the first shard from each listed family only. A complete modelling audit and temporal alignment checks are still required.


Parquet is a file format for tables. Its metadata records information such as column names and row counts. Reading metadata is much cheaper than loading billions of observations into memory. A row is one recorded observation; it is not necessarily one independent match. Many rows can belong to the same match.

A shard is one part of a large table saved in a separate file. Inspecting a shard helps us discover problems, but it cannot prove that every other shard has the same quality. Missingness means an expected value is absent. A populated field can still contain an uninformative placeholder, so we check both missing values and empty text.


In [3]:
audit_path = ROOT / 'outputs/results/D_data_audit.json'  # Locate the acquisition evidence already produced by the pipeline.
if not audit_path.exists():  # A fresh local acquisition needs an initial audit.
    audit()  # Scan downloaded raw files only when no saved audit exists.
else:  # A public clone includes compact evidence but excludes the raw dataset.
    print('Reading the saved acquisition audit; run src/audit_dota2.py explicitly to rescan raw data.')  # Explain cached execution.
report_path = ROOT / 'outputs/results/D_data_audit.json'  # Locate the newly generated audit result.
report = json.loads(report_path.read_text(encoding='utf-8'))  # Read the measurements back from the saved result.
rows = []  # Start an empty list; each entry will describe one family of data tables.
for table_name, information in report['tables'].items():  # Visit each table family and its measured metadata.
    row = {'table': table_name}  # Give this summary row a readable table name.
    for field in ['files', 'rows', 'bytes']:  # Select file count, observation count and storage size.
        row[field] = information[field]  # Copy the observed value rather than typing a number into the notebook.
    rows.append(row)  # Add the completed row to the list of table summaries.
display(pd.DataFrame(rows))  # Display the list as a table for comparison.
match_fields = ['rows', 'unique_matches', 'duplicate_ids', 'first_match_utc', 'last_match_utc']  # Select basic coverage checks.
display(pd.Series({key: report['matches'][key] for key in match_fields}))  # Show the actual metadata coverage.


Reading the saved acquisition audit; run src/audit_dota2.py explicitly to rescan raw data.


,table,files,rows,bytes
0,abilities,248,1139617029,526644564
1,actions,939,27914839,281287536
2,buildings,939,415890156,345432754
3,events,939,295640663,2425483479
4,matches,1,9385,24906542
5,modifiers,939,187395543,1679442408
6,objectives,939,145264,6558459
7,ticks,939,189040980,4352977793
8,wards,939,761616,11984637


rows                                    9385
unique_matches                          9385
duplicate_ids                              0
first_match_utc    2024-02-11 16:12:30+00:00
last_match_utc     2025-12-01 12:41:09+00:00
dtype: object

In [4]:
for family, sample in report['samples'].items():  # Examine the first downloaded shard from each audited family.
    print(family, 'sample rows:', sample['rows'], 'sample matches:', sample['unique_matches'])  # State the scope of this check.
    missing = pd.Series(sample['missing_by_column'])  # Turn the missing value counts into a labelled list.
    incomplete_columns = missing[missing > 0]  # Retain columns where at least one sampled value is missing.
    display(incomplete_columns.rename('missing cells in sampled shard').to_frame())  # Display counts without implying a full dataset audit.
print('Cautions:')  # Introduce the limitations so a reader does not overinterpret the preceding tables.
for limitation in report['limitations']:  # Visit each recorded interpretation warning.
    print('*', limitation)  # Print the warning directly from the generated result file.


ticks sample rows: 241950 sample matches: 10


,missing cells in sampled shard
respawn_seconds,218184
str,19590
agi,21293
int_,19411
armor,109755
move_speed,17258
item_0,27009
item_1,31302
item_2,38860
item_3,39685


wards sample rows: 1209 sample matches: 10


,missing cells in sampled shard
slot,37
target,414
team,414
owner_slot,821
entity_name,414
item,795
x,1209
y,1209


objectives sample rows: 206 sample matches: 10


...

actions sample rows: 35042 sample matches: 10


,missing cells in sampled shard
slot,3458
target,4917
event_subtype,34720
ability,19479
item,15822
ability_level,4917
x,35042
y,35042


buildings sample rows: 532290 sample matches: 10


,missing cells in sampled shard
hp,120230


Cautions:
* Detailed null and sampling checks cover the first shard of each listed family only, not every row.
* Ward and objective coordinates can be absent; do not assume spatial vision coverage can be reconstructed.
* Generic item strings occur; detailed item interpretation requires further validation.
* Team semantics differ between event types and must be checked against source code.
* This acquisition audit does not fit models or evaluate predictions; later study stages are recorded separately.


## Interpretation before modelling

The presence of a column does not establish that it is a usable feature. Ward coordinates may be missing, item slots can contain generic names and event team codes require interpretation. Missing role strings must be treated as unknown, not a new hero role.

Use only records available by the chosen prediction time. Do not use final match statistics to predict an earlier outcome. All samples from one match must remain in the same partition. Movement at native sampling frequency is still an approximation, not continuous ground truth. This audit establishes data provenance and identifies issues to resolve; it does not show predictive performance.


## What the output does and does not establish

If the manifest is complete, the script has received and verified the files in the pinned source snapshot. It does not establish that the publisher collected every professional match. A row count is a size measurement, not a measure of model quality.

If a missing coordinate count equals the number of rows in the sampled ward shard, that shard cannot directly locate wards on a map. If some items are generic names, a detailed item analysis needs additional validation. Repeated observations from the same match share information; splitting them across training and test data would let the model see parts of a supposedly unseen match. This is called leakage and can make performance appear better than it is.

The next step is a modelling audit: establish valid joins between tables, align their clocks, check coverage across matches and define evaluation partitions. This notebook deliberately stops before those experimental decisions.


## What the later audit corrected

The initial acquisition audit below describes table availability. Subsequent full integrity checks found that replay elapsed time differs from the match clock and replay slot order does not reliably encode team side. Notebook 02 explains the event anchored timing and hero to draft identity corrections. These corrections were applied before the final experiment. Field completeness checks on a sampled shard are not a census of every field value.